# HTR CREMMA Medieval — Fine-tuning Kraken sur Google Colab

**Avant de lancer :**
1. `Exécution` → `Modifier le type d'exécution` → **GPU** (T4 ou A100)
2. Renseigner les secrets AWS dans la cellule 1

**Ordre d'exécution :** 0 → 1 → 2 → 3 → 6a → 6b → 7

**Pipeline :** S3 → Kraken fine-tuning → S3

In [ ]:
# Cellule 0 — Vérifier le GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'Aucun GPU détecté — activer le GPU dans Exécution > Modifier le type d\'exécution')

In [ ]:
# Cellule 1 — Clés AWS (à remplir, ne pas partager)
import os

# Coller vos clés ici (ou utiliser les Secrets Colab via le cadenas à gauche)
os.environ['AWS_ACCESS_KEY_ID']     = ''  # <-- coller ici
os.environ['AWS_SECRET_ACCESS_KEY'] = ''  # <-- coller ici
os.environ['AWS_DEFAULT_REGION']    = 'eu-west-3'

if not os.environ['AWS_ACCESS_KEY_ID']:
    # Tentative via Colab Secrets (si configuré)
    try:
        from google.colab import userdata
        os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
        os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
        print('Clés chargées depuis Colab Secrets')
    except Exception:
        print('ERREUR : renseigner les clés AWS dans cette cellule ou dans Colab Secrets')
else:
    print('Clés AWS configurées')

In [ ]:
# Cellule 2 — Installer Kraken (A100 = CUDA 12.x, pas besoin de réinstaller torch)
import subprocess, sys, importlib

# Sur A100 Colab, PyTorch est préinstallé avec CUDA 12.x — on ne le réinstalle pas
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU : {torch.cuda.get_device_name(0)} (sm_{cap[0]}{cap[1]})')
    # A100 = sm_80, supporte bf16 natif
    if cap[0] >= 8:
        print('bf16 supporté (Ampere+) — sera utilisé pour l\'entraînement')

print('\nInstallation Kraken + dépendances...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'kraken', 'iso639-lang', 'opencv-python-headless', 'lxml', 'tqdm', 'boto3',
], check=True)
print('Kraken installé')

# Vérification imports
test = subprocess.run(
    [sys.executable, '-c',
     'from kraken.lib.xml import XMLPage; '
     'from kraken.train import KrakenTrainer; '
     'print("Imports OK")'],
    capture_output=True, text=True
)
print(test.stdout.strip())
if test.returncode != 0:
    print('ERREUR IMPORT :')
    print(test.stderr[-1500:])

result = subprocess.run(['ketos', '--version'], capture_output=True, text=True)
print(f'Kraken : {result.stdout.strip() or result.stderr.strip()}')

In [ ]:
# Cellule 3 — Télécharger splits + modèle de base depuis S3
import boto3
from pathlib import Path

s3 = boto3.client('s3', region_name='eu-west-3')
BUCKET = 'htr-cremma-medieval'

Path('/content/splits').mkdir(exist_ok=True)
Path('/content/models').mkdir(exist_ok=True)

for key, dest in [
    ('splits/train.txt',                      '/content/splits/train.txt'),
    ('splits/dev.txt',                        '/content/splits/dev.txt'),
    ('splits/test.txt',                       '/content/splits/test.txt'),
    ('base-model/cremma-generic-1.0.1.mlmodel', '/content/cremma_generic.mlmodel'),
]:
    s3.download_file(BUCKET, key, dest)
    print(f'OK : {dest}')

In [ ]:
# Cellule 4 — Convertir chemins Windows → /content/
for nom in ['train.txt', 'dev.txt', 'test.txt']:
    path = f'/content/splits/{nom}'
    with open(path) as f:
        lignes = f.readlines()
    nouvelles = []
    for ligne in lignes:
        ligne = ligne.strip().replace('\\', '/')
        idx = ligne.find('preprocessed/')
        if idx != -1:
            nouvelles.append(f'/content/{ligne[idx:]}\n')
    with open(path, 'w') as f:
        f.writelines(nouvelles)
    print(f'{nom} : {len(nouvelles)} lignes')

# Vérification
with open('/content/splits/train.txt') as f:
    print('Exemple :', f.readline().strip())

In [ ]:
# Cellule 5 — Télécharger les données prétraitées depuis S3 (parallèle)
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

paginator = s3.get_paginator('list_objects_v2')
objets = []
for page in paginator.paginate(Bucket=BUCKET, Prefix='preprocessed/'):
    for obj in page.get('Contents', []):
        key = obj['Key']
        if key.endswith('/') or '.' not in key.split('/')[-1]:
            continue
        objets.append(key)

print(f'{len(objets)} fichiers à télécharger...')

def download(key):
    local_path = Path('/content') / key
    local_path.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(BUCKET, key, str(local_path))
    return key

total, erreurs = 0, 0
with ThreadPoolExecutor(max_workers=16) as ex:
    futures = {ex.submit(download, k): k for k in objets}
    for fut in as_completed(futures):
        try:
            fut.result()
            total += 1
            if total % 100 == 0:
                print(f'{total}/{len(objets)} fichiers téléchargés...')
        except Exception as e:
            erreurs += 1
            print(f'Erreur : {e}')

print(f'Terminé — {total} fichiers, {erreurs} erreurs')

In [ ]:
# Cellule 6 — Lancer l'entraînement Kraken
import subprocess

# Voir les options disponibles de ketos train
print('=== Options ketos train ===')
subprocess.run(['ketos', 'train', '--help'], capture_output=False)


In [ ]:
# Cellule 6a — Télécharger Arrow + modèle depuis S3
import os, boto3, time

s3 = boto3.client('s3', region_name='eu-west-3')
BUCKET = 'htr-cremma-medieval'

os.makedirs('/content/splits', exist_ok=True)
os.makedirs('/content/models', exist_ok=True)

for s3_key, local in [
    ('splits/train.arrow',                      '/content/splits/train.arrow'),
    ('splits/dev.arrow',                        '/content/splits/dev.arrow'),
    ('base-model/cremma-generic-1.0.1.mlmodel', '/content/cremma_generic.mlmodel'),
]:
    print(f'Téléchargement {s3_key}...')
    t0 = time.time()
    s3.download_file(BUCKET, s3_key, local)
    size = os.path.getsize(local) / 1024 / 1024
    print(f'OK : {local} ({size:.1f} MB | {time.time()-t0:.0f}s)')

with open('/content/splits/train_binary.txt', 'w') as f:
    f.write('/content/splits/train.arrow\n')
with open('/content/splits/dev_binary.txt', 'w') as f:
    f.write('/content/splits/dev.arrow\n')
print('Manifests OK')

In [ ]:
# Cellule 6b — Fine-tuning depuis cremma_generic
# T4 : --precision 16-mixed, -B 8, --workers 4
# A100 : --precision bf16-mixed, -B 16, --workers 11
import subprocess, os, time, datetime, re

try:
    import torch
    GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    cap = torch.cuda.get_device_capability(0)
    IS_AMPERE = cap[0] >= 8
except:
    GPU_NAME, IS_AMPERE = 'CPU', False

PRECISION = 'bf16-mixed' if IS_AMPERE else '16-mixed'
BATCH     = 16 if IS_AMPERE else 8
WORKERS   = 11 if IS_AMPERE else 4
print(f'GPU : {GPU_NAME} | precision={PRECISION} | batch={BATCH} | workers={WORKERS}')

with open('/content/splits/train_binary.txt', 'w') as f:
    f.write('/content/splits/train.arrow\n')
with open('/content/splits/dev_binary.txt', 'w') as f:
    f.write('/content/splits/dev.arrow\n')

cmd = [
    'ketos',
    '--device', 'cuda:0',
    '--workers', str(WORKERS),
    '--precision', PRECISION,
    '--threads', '4',
    'train',
    '-f', 'binary',
    '-t', '/content/splits/train_binary.txt',
    '-e', '/content/splits/dev_binary.txt',
    '-i', '/content/cremma_generic.mlmodel',
    '--resize', 'union',
    '-o', '/content/models/htr_cremma_ft',
    '-q', 'early',
    '-N', '50',
    '--lag', '3',
    '--min-epochs', '3',
    '-B', str(BATCH),
    '-r', '0.0001',
    '--augment',
]

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['TERM'] = 'dumb'

print('Commande :', ' '.join(cmd))
print(f'Début    : {datetime.datetime.now().strftime("%H:%M:%S")}')
print('─' * 60)

t_global = time.time()
t_epoch = time.time()
epoch_num = 0
best_acc = 0.0
best_epoch = 0

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=0, env=env)
buf = ''
while True:
    ch = proc.stdout.read(1)
    if not ch:
        break
    if ch == '\r':
        print('\r' + buf, end='', flush=True)
        buf = ''
    elif ch == '\n':
        line = buf
        print(line)

        m = re.search(r'stage\s+(\d+)', line, re.IGNORECASE)
        if m:
            epoch_num = int(m.group(1))
            t_epoch = time.time()

        m_acc = re.search(r'val_accuracy[:\s]+([\d.]+)', line, re.IGNORECASE)
        if m_acc:
            acc = float(m_acc.group(1))
            elapsed_epoch = time.time() - t_epoch
            elapsed_total = time.time() - t_global
            if acc > best_acc:
                best_acc = acc
                best_epoch = epoch_num
            print(f'  ┌─ Epoch {epoch_num:>2} | acc={acc*100:.2f}% | CER={((1-acc)*100):.2f}% | {elapsed_epoch:.0f}s | total={elapsed_total/60:.1f}min')
            print(f'  └─ Meilleur : epoch {best_epoch} → {best_acc*100:.2f}%')

        m_pat = re.search(r'patience\s+(\d+)/(\d+)', line, re.IGNORECASE)
        if m_pat:
            cur, total_p = int(m_pat.group(1)), int(m_pat.group(2))
            print(f'  ⚡ Patience : {cur}/{total_p}')

        buf = ''
    else:
        buf += ch

if buf:
    print(buf)
proc.wait()

elapsed_total = time.time() - t_global
print('─' * 60)
print(f'Fin : {datetime.datetime.now().strftime("%H:%M:%S")} | Durée : {elapsed_total/60:.1f} min')
print(f'Meilleur : epoch {best_epoch} → {best_acc*100:.2f}% acc / {(1-best_acc)*100:.2f}% CER')
print(f'Code retour : {proc.returncode}')

In [ ]:
# Cellule 7 — Uploader le modèle final sur S3
import glob, datetime
from pathlib import Path

models_dir = '/content/models'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')

mlmodels    = glob.glob(f'{models_dir}/**/*.mlmodel', recursive=True)
safetensors = glob.glob(f'{models_dir}/**/*.safetensors', recursive=True)
checkpoints = glob.glob(f'{models_dir}/**/*.ckpt', recursive=True)

print('Fichiers trouvés :')
print(f'  .mlmodel     : {mlmodels}')
print(f'  .safetensors : {safetensors}')
print(f'  .ckpt        : {checkpoints}')

uploaded = []

for f in mlmodels:
    nom = f'{Path(f).stem}_{timestamp}.mlmodel'
    s3.upload_file(f, BUCKET, f'output/{nom}')
    print(f'Uploadé : s3://{BUCKET}/output/{nom}')
    uploaded.append(nom)

if not mlmodels:
    for f in safetensors:
        nom = f'{Path(f).stem}_{timestamp}.safetensors'
        s3.upload_file(f, BUCKET, f'output/{nom}')
        print(f'Uploadé : s3://{BUCKET}/output/{nom}')
        uploaded.append(nom)

if not mlmodels and not safetensors and checkpoints:
    best = sorted(checkpoints, key=lambda p: float(Path(p).stem.split('-')[-1]) if '-' in Path(p).stem else 0, reverse=True)[0]
    nom = f'{Path(best).stem}_{timestamp}.ckpt'
    s3.upload_file(best, BUCKET, f'output/{nom}')
    print(f'Uploadé (checkpoint) : s3://{BUCKET}/output/{nom}')
    uploaded.append(nom)

if not uploaded:
    print('ERREUR : aucun modèle trouvé dans', models_dir)
    for f in Path(models_dir).rglob('*'):
        print(f'  {f}')
else:
    print(f'\nOK — {len(uploaded)} fichier(s) uploadé(s) sur S3')